# 02 — Tier 1 composition-anomaly ML model: worked example

Phase 9 walkthrough for `ml/greenwashing_risk_model.py` — the project's first classical, trained
ML component (previously `ml/greenwashing_risk_model.py` was an unimplemented stub; the shipped
score in `agents/risk_agent.py` was a hand-written linear formula, not a learned model). Updated
for Phase 9d's category encoding — see section 3 below.

**Why this model exists and what it is/isn't** — see
[docs/DATA.md](../docs/DATA.md#tier-1-composition-anomaly-model-ml) for the full writeup. Short
version: no free dataset carries a real SFDR/greenwashing label
([docs/DATA.md#ground-truth-methodology](../docs/DATA.md#ground-truth-methodology)), and the only
place a real holdings-based signal exists (Tier 2, `agents/risk_agent.py`'s `compute_gap`) only
covers 4 funds — far too small to train/evaluate a classifier credibly. This notebook instead
trains a `RandomForestClassifier` on the full ~67k-fund Tier 1 population, predicting each fund's
own **real, existing** claimed `sustainability_rating` bucket from **objective** portfolio
composition (sector/asset-class/market-cap/credit-quality/controversial-business-involvement
percentages) plus (Phase 9d) three leakage-safe `category`-derived features — not a fabricated
greenwashing label. The model's own `predict_proba` gives an anomaly signal: how atypical a fund's
real composition looks for the rating tier it claims.

This is a **different, coarser signal** from Tier 2's holdings-based gap, not a replacement for
it — the cross-check at the end of this notebook shows the two disagreeing on the same fund, and
explains why that's expected, not a bug.

In [1]:
from pathlib import Path

import pandas as pd

from greenlux_sentinel.agents import risk_agent
from greenlux_sentinel.etl import load_funds_postgres as lfp
from greenlux_sentinel.etl import load_verified_holdings_cosmos as lvhc
from greenlux_sentinel.ml import greenwashing_risk_model as model

RAW = Path("..") / "data" / "raw"

mutual_funds = lfp.transform(pd.read_csv(RAW / "morningstar_european_mutual_funds.csv", low_memory=False), "mutual_fund")
etfs = lfp.transform(pd.read_csv(RAW / "morningstar_european_etfs.csv", low_memory=False), "etf")
df = pd.concat([mutual_funds, etfs], ignore_index=True)

print(f"{len(df)} total Tier 1 funds")
print(f"{df['sustainability_rating'].notna().sum()} carry a claimed sustainability_rating (the only ones this model can be trained/evaluated on)")

67098 total Tier 1 funds
40737 carry a claimed sustainability_rating (the only ones this model can be trained/evaluated on)


## 1. Features and target — and why E/S/G subscores are excluded

Target: `sustainability_rating` (1-5 claimed globes), bucketed Low(1-2)/Medium(3)/High(4-5) by
`model._bucket_rating`.

Features: `model.FEATURE_COLUMNS` — 41 **objective** portfolio-composition columns (sector
allocation, asset-class mix, market-cap tiers, credit-quality tiers, controversial-business-
involvement percentages), **plus** (Phase 9d) three `category`-derived features,
`model.CATEGORY_RATE_COLUMNS` — see section 3. `environmental_score`/`social_score`/
`governance_score`/`sustainability_score` are deliberately **excluded** — `db/schema.sql` already
comments these as claimed-side, the same signal as the target itself. Using them as features would
make the prediction circular (the model would just learn "predict the label from a near-copy of
the label"), not a genuine test of whether objective composition predicts the claim.

In [2]:
print(f"{len(model.FEATURE_COLUMNS)} feature columns:")
print(model.FEATURE_COLUMNS)

41 feature columns:
['asset_stock', 'asset_bond', 'asset_cash', 'asset_other', 'sector_basic_materials', 'sector_consumer_cyclical', 'sector_financial_services', 'sector_real_estate', 'sector_consumer_defensive', 'sector_healthcare', 'sector_utilities', 'sector_communication_services', 'sector_energy', 'sector_industrials', 'sector_technology', 'market_cap_giant', 'market_cap_large', 'market_cap_medium', 'market_cap_small', 'market_cap_micro', 'credit_aaa', 'credit_aa', 'credit_a', 'credit_bbb', 'credit_bb', 'credit_b', 'credit_below_b', 'credit_not_rated', 'involvement_abortive_contraceptive', 'involvement_alcohol', 'involvement_animal_testing', 'involvement_controversial_weapons', 'involvement_gambling', 'involvement_gmo', 'involvement_military_contracting', 'involvement_nuclear', 'involvement_palm_oil', 'involvement_pesticides', 'involvement_small_arms', 'involvement_thermal_coal', 'involvement_tobacco']


## 2. Methodology: why a naive row-level train/test split is wrong here

`~6%` of unique ISINs in this dataset have more than one `fund_id` row (share classes of the same
underlying fund — same portfolio, same claimed rating, near-identical feature values). A plain
row-level split lets near-duplicate rows of the *same fund* land on both sides of the split, which
inflates the test-set score without the model having learned anything more general. The cell below
reproduces this live: a naive split vs. `GroupShuffleSplit` grouped by ISIN
(`model._groups` — fallback to `fund_id` where ISIN is null), same model, same data.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

X_all, y_all, _ = model.build_feature_matrix(df)
groups_all = model._groups(df.loc[X_all.index])

isin_share_classes = df.loc[X_all.index, "isin"].value_counts()
pct_multi = 100 * (isin_share_classes > 1).sum() / len(isin_share_classes)
print(f"{pct_multi:.1f}% of ISINs in the scorable set have more than one fund_id row (share classes)")

def _fit_eval(X_train, X_test, y_train, y_test, label):
    clf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                                  class_weight="balanced", random_state=42)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1 = f1_score(y_test, pred, average="macro")
    print(f"{label}: accuracy={acc:.4f}  macro_f1={f1:.4f}")
    return acc, f1

# Naive row-level split -- ignores which rows share an ISIN.
Xtr, Xte, ytr, yte = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
naive_acc, naive_f1 = _fit_eval(Xtr, Xte, ytr, yte, "naive row-level split (inflated, don't trust this)")

# Group split by ISIN -- the methodologically correct one, and what ml/greenwashing_risk_model.py
# actually ships.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_all, y_all, groups=groups_all))
group_acc, group_f1 = _fit_eval(X_all.iloc[train_idx], X_all.iloc[test_idx],
                                 y_all.iloc[train_idx], y_all.iloc[test_idx],
                                 "ISIN-grouped split (honest, what's actually shipped)")

print(f"\ninflation from ignoring share-class grouping: {(naive_acc - group_acc) * 100:.1f} accuracy points")

6.2% of ISINs in the scorable set have more than one fund_id row (share classes)


naive row-level split (inflated, don't trust this): accuracy=0.9283  macro_f1=0.9297


ISIN-grouped split (honest, what's actually shipped): accuracy=0.9019  macro_f1=0.9034

inflation from ignoring share-class grouping: 2.6 accuracy points


The gap above is small but real — and it's reported here as evidence the group split is the
right methodological call, not hidden. `ml/greenwashing_risk_model.train()` also fits the
imputation medians and (Phase 9d, next section) the category-rate encoding on the train fold only
and applies them unchanged to the test fold (see its `build_feature_matrix(df, fit_stats=...)`
parameter) — a second, smaller leakage source a naive `fillna(df.median())` before splitting would
introduce, avoided here too.

## 3. Category encoding (Phase 9d)

`category` (295 distinct Morningstar categories, e.g. "US Large-Cap Blend Equity") was originally
left out of v1 — too high-cardinality for naive one-hot, and target-mean encoding risks leakage
unless done carefully (docs/DATA.md's deferred-follow-up note). Added here as three features,
`category_low_rate`/`category_medium_rate`/`category_high_rate`
(`model.CATEGORY_RATE_COLUMNS`): for each category, the Laplace-smoothed empirical rate of
Low/Medium/High claimed ratings among *other* funds in that same category — computed on the
**train fold only** (`model._fit_category_rates()`) and applied unchanged to the test fold and at
score() time (`model._apply_category_rates()`), the exact same fit-on-train/apply-to-test
discipline already used for median imputation above. A category with too few training-fold funds
to trust its own rate is smoothed toward the global training-set rate
(`_CATEGORY_SMOOTHING = 5.0` "pseudo-funds" of weight); a category never seen in training falls
back to the global rate entirely.

In [4]:
from greenlux_sentinel.ml.greenwashing_risk_model import _fit_category_rates

y_demo = df["sustainability_rating"].map(model._bucket_rating)
category_demo = df.loc[y_demo.notna(), "category"].fillna("__missing_category__")
y_demo = y_demo.loc[y_demo.notna()]
rate_table = _fit_category_rates(category_demo, y_demo)

print("Global training-set rate (fallback for rare/unseen categories):")
print(rate_table["__global__"])
print()
for cat in ["US Large-Cap Blend Equity", "Sector Equity Alternative Energy", "EUR High Yield Bond"]:
    if cat in rate_table:
        print(f"{cat}: {rate_table[cat]}")

Global training-set rate (fallback for rare/unseen categories):
{'Low': np.float64(0.2561307901907357), 'Medium': np.float64(0.3840243513268036), 'High': np.float64(0.35984485848246067)}

US Large-Cap Blend Equity: {'Low': np.float64(0.22196436469669586), 'Medium': np.float64(0.46020048213536), 'High': np.float64(0.3178351531679441)}
Sector Equity Alternative Energy: {'Low': np.float64(0.02065570888634965), 'Medium': np.float64(0.030969705752161583), 'High': np.float64(0.9483745853614888)}
EUR High Yield Bond: {'Low': np.float64(0.20932402672539285), 'Medium': np.float64(0.7112345446114249), 'High': np.float64(0.07944142866318223)}


## 4. Train the real shipped model (now including category encoding) and report full metrics

In [5]:
result = model.train(df)
m = result.metrics

print(f"n_train={m['n_train']}  n_test={m['n_test']}")
print(f"accuracy={m['accuracy']:.4f}  macro_f1={m['macro_f1']:.4f}")
print(f"\nconfusion matrix {m['confusion_matrix_labels']}:")
for label, row in zip(m["confusion_matrix_labels"], m["confusion_matrix"], strict=True):
    print(f"  {label:>8}: {row}")

print("\nper-class precision/recall/F1:")
for label in m["confusion_matrix_labels"]:
    r = m["classification_report"][label]
    print(f"  {label:>8}: precision={r['precision']:.3f}  recall={r['recall']:.3f}  f1={r['f1-score']:.3f}  support={int(r['support'])}")

baseline_acc = y_all.value_counts(normalize=True).max()
print(f"\nmost-frequent-class baseline accuracy: {baseline_acc:.4f} (model clears this by {(m['accuracy'] - baseline_acc) * 100:.1f} points)")

n_train=32631  n_test=8106
accuracy=0.9024  macro_f1=0.9038

confusion matrix ['Low', 'Medium', 'High']:
       Low: [1856, 102, 49]
    Medium: [135, 2779, 252]
      High: [65, 188, 2680]

per-class precision/recall/F1:
       Low: precision=0.903  recall=0.925  f1=0.914  support=2007
    Medium: precision=0.906  recall=0.878  f1=0.891  support=3166
      High: precision=0.899  recall=0.914  f1=0.906  support=2933

most-frequent-class baseline accuracy: 0.3840 (model clears this by 51.8 points)


Neither number is suspiciously perfect (not 0.99+, which would itself be a red flag for
leakage) — a credible, non-trivial signal, not noise. **Honest result, not spun as a clean win**:
adding category encoding moved held-out accuracy from 90.94%/0.9105 (Phase 9, no category) to
90.24%/0.9038 (Phase 9d, with category) — essentially flat, arguably a hair worse on this single
split. The category-rate features turn out to be the **top 3 most important features in the whole
model** (see the next cell) despite that flat headline number — plausibly because they're
partially redundant with the sector/asset-class composition features already in the model
(a fund's category correlates with its typical sector tilt), so the trees reallocate importance to
them without a corresponding gain in held-out accuracy. Reported exactly as measured, matching
this project's existing practice of not overselling a result (see docs/DATA.md's regression-vs-
classification note from Phase 9 for the same honesty standard applied there).

## 5. What is the model actually keying on? (feature importances)

In [6]:
import pandas as pd

# feature_importances_ is ordered [imputed features..., __missing flags...] -- reconstruct the real column names.
X_cols = model.build_feature_matrix(df)[0].columns
importances = pd.Series(result.model.feature_importances_, index=X_cols).sort_values(ascending=False)

_EXPLAIN = {
    "sector_energy": "heavier energy-sector tilt",
    "sector_basic_materials": "heavier basic-materials tilt",
    "sector_technology": "heavier technology tilt",
    "sector_communication_services": "heavier communication-services tilt",
    "involvement_thermal_coal": "more thermal-coal business involvement",
    "involvement_animal_testing": "more animal-testing business involvement",
    "involvement_pesticides": "more pesticides business involvement",
    "market_cap_small": "more small-cap exposure",
    "market_cap_medium": "more mid-cap exposure",
    "market_cap_giant": "more giant-cap (mega-cap) exposure",
}

print("Top 10 features by importance:")
for col, val in importances.head(10).items():
    note = _EXPLAIN.get(col, "")
    print(f"  {col:<32} {val:.4f}  {note}")

Top 10 features by importance:
  category_low_rate                0.0780  
  category_high_rate               0.0707  
  category_medium_rate             0.0422  
  sector_energy                    0.0394  heavier energy-sector tilt
  market_cap_small                 0.0382  more small-cap exposure
  market_cap_giant                 0.0362  more giant-cap (mega-cap) exposure
  involvement_thermal_coal         0.0313  more thermal-coal business involvement
  market_cap_medium                0.0285  more mid-cap exposure
  involvement_pesticides           0.0277  more pesticides business involvement
  sector_technology                0.0272  heavier technology tilt


These are intuitively sane — the three category-rate features dominate (unsurprising: a
fund's Morningstar category is a strong prior for what rating funds like it typically claim), and
beneath them, sector tilt (energy, technology) and controversial-business involvement (thermal
coal, pesticides) are well-known, real ESG-score drivers. That the model's top features line up
with domain intuition is itself a sanity check that it learned a real pattern, not noise — even
though, per the honest note above, the category features' high individual importance didn't
translate into a clear held-out accuracy improvement.

## 6. One concrete fund, start to finish

In [7]:
fund_row = df.loc[df["fund_id"] == "0P00018CYB"].iloc[0]
print(fund_row[["fund_id", "isin", "name", "sustainability_rating", "sustainability_score"]])

print("\nreal composition values (non-zero only):")
for col in model.FEATURE_COLUMNS:
    v = fund_row[col]
    if pd.notna(v) and v != 0:
        print(f"  {col:<32} {v}")

fund_id                                                0P00018CYB
isin                                                 IE00BYVJRR92
name                     iShares MSCI USA SRI UCITS ETF USD (Acc)
sustainability_rating                                         5.0
sustainability_score                                         20.0
Name: 63112, dtype: object

real composition values (non-zero only):
  asset_stock                      99.82
  asset_bond                       0.02
  asset_cash                       0.15
  sector_basic_materials           3.07
  sector_consumer_cyclical         17.92
  sector_financial_services        12.44
  sector_real_estate               4.76
  sector_consumer_defensive        11.1
  sector_healthcare                17.43
  sector_utilities                 0.93
  sector_communication_services    4.82
  sector_energy                    0.67
  sector_industrials               11.47
  sector_technology                15.38
  market_cap_giant                 

In [8]:
out = model.score(result, fund_row.to_dict())
for k, v in out.items():
    if k != "caveat":
        print(f"{k}: {v}")

predicted_rating_bucket: High
actual_rating_bucket: High
composition_anomaly_score: 16.3
composition_anomaly_tier: Low
model_version: tier1-composition-anomaly-v2


**Reading this**: the model predicts `High` and the fund's real claimed bucket is also
`High` — a correct prediction — and the composition_anomaly_score is low (well under the
`Medium`/`High` anomaly thresholds `result.tier_thresholds`). In plain language: this fund's
objective portfolio composition (large-cap US equity, near-zero energy/thermal-coal exposure,
0.67% `sector_energy`, 24.56% `involvement_animal_testing`) looks entirely typical for a fund
claiming a High sustainability rating — nothing about its *sector mix* raises a flag.

**Contrast** — a fund the model does flag, for comparison:

In [9]:
scorable = df.loc[df["sustainability_rating"].notna()]
flagged = None
for _, row in scorable.sample(3000, random_state=1).iterrows():
    out2 = model.score(result, row.to_dict())
    if out2["composition_anomaly_tier"] == "High" and out2["actual_rating_bucket"] == "High":
        flagged = (row, out2)
        break

flagged_row, flagged_out = flagged
print(flagged_row[["fund_id", "isin", "name", "category", "sustainability_rating"]])
print()
for k, v in flagged_out.items():
    if k != "caveat":
        print(f"{k}: {v}")

fund_id                                                         F0GBR067HC
isin                                                          IE00B0JY6J37
name                     PineBridge Global Funds - US Large Cap Researc...
category                                         US Large-Cap Blend Equity
sustainability_rating                                                  4.0
Name: 55662, dtype: object

predicted_rating_bucket: Medium
actual_rating_bucket: High
composition_anomaly_score: 66.76
composition_anomaly_tier: High
model_version: tier1-composition-anomaly-v2


This fund claims a `High` sustainability rating, but the model predicts `Medium` from its
objective composition alone — a real, non-cherry-picked example (found by scanning a random sample
of the real scorable population) of a fund whose *portfolio-composition profile* looks more
consistent with a lower claimed tier than the one it actually claims.

## 6. Cross-check against Tier 2's holdings-based gap (the honest disagreement finding)

Does this Tier 1, composition-based signal agree with Tier 2's real, security-level holdings-based
`compute_gap()` for the same fund? Reusing `etl.load_verified_holdings_cosmos` directly against
the local `data/raw/verified_holdings/*.csv` + `public_company_esg_ratings.csv` — no live Cosmos
DB needed, since `transform()` is a pure function.

In [10]:
holdings = lvhc.load_raw_holdings(RAW / "verified_holdings")
esg = pd.read_csv(RAW / "public_company_esg_ratings.csv")
docs = {d["isin"]: d for d in lvhc.transform(holdings, esg)}

isin = fund_row["isin"]
tier2_doc = docs[isin]
tier2_gap = risk_agent.compute_gap(fund_row["sustainability_rating"], tier2_doc["holdings_implied_esg_score"])

print(f"fund: {fund_row['name']} ({isin})")
print(f"Tier 1 composition-anomaly score : {out['composition_anomaly_score']:.2f}  (tier: {out['composition_anomaly_tier']})")
print(f"Tier 2 holdings-based gap        : {tier2_gap:.2f}  (holdings_implied_esg={tier2_doc['holdings_implied_esg_score']})")

fund: iShares MSCI USA SRI UCITS ETF USD (Acc) (IE00BYVJRR92)
Tier 1 composition-anomaly score : 16.30  (tier: Low)
Tier 2 holdings-based gap        : 53.03  (holdings_implied_esg=1039.64)


**The two signals do not agree** — Tier 1's composition-anomaly score is low (this fund's
*sector/asset/involvement mix* looks like a typical High-claiming fund), while Tier 2's real
holdings-based gap is large (this fund's *actual constituent-level ESG scores*, security by
security, tell a less flattering story than the claimed rating).

**This is the expected, correct result, not a bug.** The two models answer genuinely different
questions:

- **Tier 1 (this model)**: "Is this fund's claim typical for funds with this kind of *portfolio
  composition* (sector mix, asset allocation, controversial-involvement levels), across the whole
  population?" — broad coverage (~41k funds), but coarse: composition can look completely normal
  even when specific constituents are individually weak on ESG.
- **Tier 2 (`risk_agent.compute_gap`)**: "Given this fund's *actual, named constituent holdings*
  and their real, individual ESG scores, does the weighted-average holdings profile support the
  claimed rating?" — the more grounded question, but only answerable for the 4 funds with real
  security-level holdings data.

This is exactly *why* Tier 2 exists at all, and why this project keeps the two-tier architecture
(`CLAUDE.md` decision #2) rather than collapsing them: a population-relative composition check and
a real security-level holdings check can legitimately disagree, and a reader should see both, not
just one.

## Caveats

Same framing as `risk_agent.CAVEAT`, restated for this model
(`model.CAVEAT`): this is a data-driven proxy for whether a claimed rating looks statistically
typical given objective portfolio composition — **not** a determination of greenwashing, SFDR
non-compliance, or any other regulatory finding. Portfolio-scope evaluation only: no drift
monitoring, no retraining pipeline. Trained/evaluated against the local `data/raw/*.csv` files;
also run live against the real deployed Postgres since Phase 9b (see docs/PROGRESS_LOG.md).